In [2]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELSDIR  = CONFIGS['filepaths']['models']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'

# SR-gauss fieldvars are kernel-integrated; SR-BL uses direct bl
SR_GAUSS_FIELDVARS = MODELS['sr']['runs']['sr_gauss']['fieldvars']  # ['rh','thetae','thetaestar']
NN_SEEDS           = MODELS['nn']['seeds']                           # seeds for kernel weight averaging

COLORS = {'pod_bl': MODELS['pod']['runs']['pod_bl']['color']}
LABELS = {'pod_bl': MODELS['pod']['runs']['pod_bl']['description']}
for name, cfg in MODELS['nn']['runs'].items():
    COLORS[name] = cfg['color']
for name, cfg in MODELS['sr']['optimizedeqs'].items():
    COLORS[name] = cfg['color']
    LABELS[name] = cfg['description']

MINSAMP = 30   # minimum samples per bin
NBINS   = 40   # 1-D bin count

In [ ]:
# ── Stats ─────────────────────────────────────────────────────────────────────
with open(os.path.join(SPLITSDIR, 'stats.json'), 'r', encoding='utf-8') as f:
    STATS = json.load(f)
tp_mean = float(STATS['tp_mean'])
tp_std  = float(STATS['tp_std'])

# ── Normalised test split ──────────────────────────────────────────────────────
with xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf') as _ds:
    ntime = _ds.sizes['time']; nlat = _ds.sizes['lat']; nlon = _ds.sizes['lon']
    nsig  = _ds.sizes.get('sig', 1)
    refshape = (ntime, nlat, nlon)

    lats = _ds.lat.values; lons = _ds.lon.values
    dsig = _ds['dsig'].values                                   # (nsig,)

    levmin, levmax = CONFIGS['domain']['levrange']
    sig_hpa = np.linspace(levmin, levmax, nsig)                 # pressure axis for kernel plots

    # Field stacks for kernel integration (nsamples, nfieldvars, nsig)
    _farrs = [_ds[v].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
              for v in SR_GAUSS_FIELDVARS]
    fieldstack  = np.stack(_farrs, axis=1)
    surfmask    = (_ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
                   if 'surfmask' in _ds else None)

    def _ravel(da):
        """Ravel a DataArray to (ntime*nlat*nlon,), tiling if no time dim."""
        if 'time' in da.dims:
            return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values, (ntime, 1, 1)).ravel()

    # Scalar local vars (standardised) — tile time-invariant fields
    lf_norm  = _ravel(_ds['lf'])  if 'lf'  in _ds else None
    shf_norm = _ravel(_ds['shf']) if 'shf' in _ds else None
    lhf_norm = _ravel(_ds['lhf']) if 'lhf' in _ds else None

    # BL: direct if scalar, mean over sig if profile
    if 'bl' in _ds:
        _bl = _ds['bl']
        bl_norm = (_ravel(_bl.mean('sig')) if 'sig' in _bl.dims else _ravel(_bl))
    else:
        bl_norm = None

# ── Kernel integration (matching train.py) ────────────────────────────────────
def kernel_integrate(fields, weights, dsig, mask=None):
    w = fields * weights[None, :, :] * dsig[None, None, :]
    if mask is not None:
        w = w * mask[:, None, :]
    return w.sum(axis=2)                                        # (N, nfieldvars)

# Load NN-GAUSS kernel weights (average across seeds)
_kw_list = []
for _seed in NN_SEEDS:
    _wpath = os.path.join(WEIGHTSDIR, f'nn_gauss_{_seed}_weights.nc')
    if os.path.exists(_wpath):
        with xr.open_dataset(_wpath, engine='h5netcdf') as _wds:
            _kw_list.append(_wds['k'].values)                   # (nfieldvars, nsig)

if _kw_list:
    avg_kernel = np.mean(_kw_list, axis=0)                      # (nfieldvars, nsig) — for plotting
    _ki = np.mean([kernel_integrate(fieldstack, kw, dsig, surfmask) for kw in _kw_list], axis=0)
    rh_k         = _ki[:, 0]
    thetae_k     = _ki[:, 1]
    thetaestar_k = _ki[:, 2]
    print(f'Kernel weights: {len(_kw_list)}/{len(NN_SEEDS)} seeds loaded')
else:
    avg_kernel   = None
    rh_k         = fieldstack[:, 0, :].mean(axis=1)
    thetae_k     = fieldstack[:, 1, :].mean(axis=1)
    thetaestar_k = fieldstack[:, 2, :].mean(axis=1)
    print('Warning: no kernel weights found, falling back to vertical mean')

# ── Native test split (true precip + land mask) ───────────────────────────────
with xr.open_dataset(os.path.join(SPLITSDIR, f'{SPLIT}.h5'), engine='h5netcdf') as _ds:
    true_flat = _ds.tp.transpose('time','lat','lon').values.ravel()
    if 'lf' in _ds:
        _lf2d       = _ds['lf'].squeeze() if 'time' not in _ds['lf'].dims else _ds['lf'].isel(time=0)
        land_mask_2d = (_lf2d.values >= 0.5)                    # True = land
        land_flat    = np.broadcast_to(land_mask_2d[None], refshape).ravel().copy()
    else:
        land_mask_2d = land_flat = None

# ── SR registry ────────────────────────────────────────────────────────────────
_regpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
if os.path.exists(_regpath):
    with open(_regpath, 'rb') as _f:
        SR_REGISTRY = pickle.load(_f)
    print(f'SR registry: {list(SR_REGISTRY.keys())}')
else:
    SR_REGISTRY = {}
    print('Warning: SR registry not found')

# ── POD-BL ────────────────────────────────────────────────────────────────────
_pod_path = os.path.join(MODELSDIR, 'pod', 'pod_bl.npz')
if os.path.exists(_pod_path):
    with np.load(_pod_path) as _d:
        pod_alpha = float(_d['alpha']); pod_xcrit = float(_d['xcrit'])
    print(f'POD-BL: alpha={pod_alpha:.4f}, xcrit={pod_xcrit:.4f}')
else:
    pod_alpha = pod_xcrit = None

valid = (np.isfinite(true_flat) & np.isfinite(rh_k) &
         np.isfinite(thetae_k) & np.isfinite(thetaestar_k))
print(f'Valid samples: {valid.sum():,}')

In [5]:
SRFUNCTIONS = {
    'cube':lambda x:x**3, 'square':lambda x:x**2, 'neg':lambda x:-x,
    'sqrt':np.sqrt, 'exp':np.exp, 'log':np.log, 'abs':np.abs,
    'sin':np.sin, 'cos':np.cos, 'max':np.maximum, 'min':np.minimum}

def to_phys(sr_out):
    """SR latent output → physical precipitation (mm / 3 h)"""
    return np.expm1(tp_std * np.maximum(0.0, np.asarray(sr_out, dtype=float)))

def eval_sr(name, var_dict):
    """Evaluate SR equation and return physical precipitation array."""
    if name not in SR_REGISTRY:
        return None
    entry = SR_REGISTRY[name]
    ns    = dict(SRFUNCTIONS, __builtins__={})
    ns.update(entry['constants'])
    ns.update(var_dict)
    out = eval(entry['form'], ns)
    if np.ndim(out) == 0:
        n   = next((len(v) for v in var_dict.values() if hasattr(v, '__len__')), 1)
        out = np.full(n, float(out))
    return to_phys(np.asarray(out, dtype=float))

def bin1d(x, z, nbins=NBINS, minsamp=MINSAMP, plo=1, phi=99):
    """1-D conditional mean of z given x; returns (edges, centers, means, counts)."""
    finite = np.isfinite(x) & np.isfinite(z)
    x, z   = x[finite], z[finite]
    edges  = np.unique(np.percentile(x, np.linspace(plo, phi, nbins + 1)))
    n      = len(edges) - 1
    xi     = np.clip(np.digitize(x, edges) - 1, 0, n - 1)
    means  = np.full(n, np.nan); counts = np.zeros(n, int)
    for i in range(n):
        sel = xi == i; counts[i] = sel.sum()
        if counts[i] >= minsamp: means[i] = z[sel].mean()
    return edges, 0.5*(edges[:-1]+edges[1:]), means, counts

def bin2d(x, y, z, xedges, yedges, minsamp=MINSAMP):
    """Fast 2-D binned mean of z; returns (means [ny,nx], counts)."""
    finite  = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[finite], y[finite], z[finite]
    nx, ny  = len(xedges)-1, len(yedges)-1
    xi = np.clip(np.digitize(x, xedges)-1, 0, nx-1)
    yi = np.clip(np.digitize(y, yedges)-1, 0, ny-1)
    idx    = yi*nx + xi
    counts = np.bincount(idx, minlength=nx*ny).reshape(ny, nx)
    sums   = np.bincount(idx, weights=z, minlength=nx*ny).reshape(ny, nx)
    return np.where(counts >= minsamp, sums/counts, np.nan), counts

def prange(arr, lo=1, hi=99):
    """Percentile range of finite values."""
    a = arr[np.isfinite(arr)]
    return float(np.percentile(a, lo)), float(np.percentile(a, hi))

In [6]:
# ── SR-MED regime statistics ──────────────────────────────────────────────────
if 'sr_med' in SR_REGISTRY:
    _c = SR_REGISTRY['sr_med']['constants']
    a_m, b_m, c_m = _c['a'], _c['b'], _c['c']
    M_all = rh_k[valid]; I_all = thetae_k[valid] - b_m*thetaestar_k[valid] - c_m
    P_all = true_flat[valid]
    moist = M_all >= I_all; insta = ~moist
    print('SR-MED regimes (test set):')
    print(f'  Moisture-controlled (M≥I): {100*moist.mean():.1f}%  mean P={P_all[moist].mean():.3f} mm')
    print(f'  Instability-ctrl   (I>M):  {100*insta.mean():.1f}%  mean P={P_all[insta].mean():.3f} mm')
    if land_flat is not None:
        lv = land_flat[valid].astype(bool)
        for lbl, lmask in [('Land', lv), ('Ocean', ~lv)]:
            for rlbl, rmask in [('Moisture', moist), ('Instability', insta)]:
                m = lmask & rmask
                print(f'    {lbl:<5} {rlbl:<12}: {100*m.sum()/lmask.sum():.1f}%  mean P={P_all[m].mean():.3f}')

# ── SHF correlations ───────────────────────────────────────────────────────────
if shf_norm is not None:
    print('\nCorrelations with standardised SHF:')
    for lbl, arr in [('LHF~',lhf_norm),('LF~',lf_norm),('RH_hat',rh_k),('θe_hat',thetae_k),('Precip',true_flat)]:
        if arr is not None:
            r = np.corrcoef(shf_norm[valid], arr[valid])[0,1]
            print(f'  r(SHF, {lbl:10s}) = {r:+.3f}')
    if lhf_norm is not None:
        bowen = shf_norm[valid] / (lhf_norm[valid] + 1e-6)
        print(f'\n  High-SHF samples (top 25%):')
        shf_hi = shf_norm[valid] >= np.percentile(shf_norm[valid], 75)
        print(f'    mean LHF~={lhf_norm[valid][shf_hi].mean():.3f}  mean RH_hat={rh_k[valid][shf_hi].mean():.3f}  mean P={P_all[shf_hi].mean():.3f}')
        if land_flat is not None:
            lv = land_flat[valid].astype(bool)
            for lbl, lmask in [('Land',lv),('Ocean',~lv)]:
                m = lmask & shf_hi
                if m.sum() > 0:
                    print(f'    {lbl}: mean P={P_all[m].mean():.3f}  mean SHF~={shf_norm[valid][m].mean():.3f}')

SR-MED regimes (test set):
  Moisture-controlled (M≥I): 24.7%  mean P=1.079 mm
  Instability-ctrl   (I>M):  75.3%  mean P=0.668 mm
    Land  Moisture    : 37.8%  mean P=1.398
    Land  Instability : 62.2%  mean P=0.833
    Ocean Moisture    : 19.1%  mean P=0.812
    Ocean Instability : 80.9%  mean P=0.615

Correlations with standardised SHF:
  r(SHF, LHF~      ) = +0.267


IndexError: boolean index did not match indexed array along dimension 0; dimension is 651 but corresponding boolean dimension is 1437408

In [ ]:
# ── Fig A: SR-MED  M–I phase space ───────────────────────────────────────────
if 'sr_med' not in SR_REGISTRY:
    print('sr_med not in registry — skipping')
else:
    _c = SR_REGISTRY['sr_med']['constants']
    a_m, b_m, c_m = _c['a'], _c['b'], _c['c']

    M = rh_k[valid];  I = thetae_k[valid] - b_m*thetaestar_k[valid] - c_m
    P = true_flat[valid]

    M_lo, M_hi = prange(M);  I_lo, I_hi = prange(I)
    axlo = min(M_lo, I_lo);   axhi = max(M_hi, I_hi)

    # shared edges for binned panels
    NB = 35
    xe = np.linspace(M_lo, M_hi, NB+1); ye = np.linspace(I_lo, I_hi, NB+1)
    xcen = 0.5*(xe[:-1]+xe[1:]); ycen = 0.5*(ye[:-1]+ye[1:])

    obs_bin, _  = bin2d(M, I, P, xe, ye)
    sr_pred     = to_phys(a_m * np.maximum(M, I)**3)
    pred_bin, _ = bin2d(M, I, sr_pred, xe, ye)
    density, _  = np.histogram2d(M, I, bins=[xe, ye])
    density     = np.log10(density.T.clip(1))

    vmax = float(np.nanpercentile(np.r_[obs_bin[~np.isnan(obs_bin)],
                                        pred_bin[~np.isnan(pred_bin)]], 97))

    # analytic grid for SR-MED panel
    NG = 200
    Mg, Ig = np.meshgrid(np.linspace(M_lo, M_hi, NG), np.linspace(I_lo, I_hi, NG))
    P_grid = to_phys(a_m * np.maximum(Mg, Ig)**3)

    fig, axs = pplt.subplots(ncols=3, figwidth=7.5, share=False)
    kw = dict(cmap='Blues', vmin=0, vmax=vmax)

    # (a) SR-MED analytic prediction
    m0 = axs[0].pcolormesh(np.linspace(M_lo,M_hi,NG), np.linspace(I_lo,I_hi,NG), P_grid, **kw)
    axs[0].plot([axlo,axhi],[axlo,axhi],'k--',lw=1,label='M = I')
    axs[0].format(xlabel=r'M ($\widehat{\mathrm{RH}}$)', ylabel=r'I (instability)',
                  title='SR-MED predicted', xlim=(M_lo,M_hi), ylim=(I_lo,I_hi))

    # (b) Observed binned P
    m1 = axs[1].pcolormesh(xcen, ycen, obs_bin, **kw)
    axs[1].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
    axs[1].format(xlabel=r'M', ylabel=r'I', title='ERA5 observed (binned)',
                  xlim=(M_lo,M_hi), ylim=(I_lo,I_hi))

    # (c) Sample density
    m2 = axs[2].pcolormesh(xcen, ycen, density, cmap='Grays', vmin=0)
    axs[2].contour(xcen, ycen, obs_bin, levels=5, colors='C0', linewidths=0.7, alpha=0.7)
    axs[2].plot([axlo,axhi],[axlo,axhi],'r--',lw=1,label='M = I')
    axs[2].format(xlabel=r'M', ylabel=r'I', title='Sample density (log₁₀)',
                  xlim=(M_lo,M_hi), ylim=(I_lo,I_hi))

    fig.colorbar(m0, ax=axs[:2], loc='b', label='Mean P (mm / 3 h)')
    fig.colorbar(m2, ax=axs[2],  loc='b', label='log₁₀(N)')
    fig.format(abc=True)
    pplt.show()
    fig.save('../figs/fig_srmed.pdf'); fig.save('../figs/fig_srmed.png')
    print(f'Moisture-ctrl: {100*(M>=I).mean():.1f}%  mean obs P={P[M>=I].mean():.3f} mm')
    print(f'Instability:   {100*(I>M).mean():.1f}%  mean obs P={P[I>M].mean():.3f} mm')

In [ ]:
# ── Fig B: SR-HI  T–S surface-correction plot ────────────────────────────────
if 'sr_hi' not in SR_REGISTRY or shf_norm is None or lf_norm is None:
    print('sr_hi not in registry or surface vars missing — skipping')
else:
    _c = SR_REGISTRY['sr_hi']['constants']
    a_h, b_h, c_h, d_h = _c['a'], _c['b'], _c['c'], _c['d']

    M_hi = rh_k[valid]; I_hi = thetae_k[valid] + b_h*thetaestar_k[valid] + c_h
    T    = np.maximum(M_hi, I_hi)                          # thermodynamic term (pre-scaling)
    S    = np.maximum(lf_norm[valid], shf_norm[valid])     # surface suppression term
    P    = true_flat[valid]

    T_lo, T_hi = prange(T); S_lo, S_hi = prange(S)
    NB = 35
    Te = np.linspace(T_lo, T_hi, NB+1); Se = np.linspace(S_lo, S_hi, NB+1)
    Tcen = 0.5*(Te[:-1]+Te[1:]); Scen = 0.5*(Se[:-1]+Se[1:])

    obs_ts,  _ = bin2d(T, S, P, Te, Se)
    sr_pred_ts = to_phys((a_h*T + d_h*S)**3)
    pred_ts, _ = bin2d(T, S, sr_pred_ts, Te, Se)

    # analytic grid
    NG = 150
    Tg, Sg = np.meshgrid(np.linspace(T_lo,T_hi,NG), np.linspace(S_lo,S_hi,NG))
    P_grid = to_phys((a_h*Tg + d_h*Sg)**3)
    vmax = float(np.nanpercentile(obs_ts[~np.isnan(obs_ts)], 97))

    # partial dependence: P vs S at fixed T percentiles
    S_sw  = np.linspace(S_lo, S_hi, 200)
    T_pcts = [25, 50, 75, 90]
    T_fixed = np.percentile(T, T_pcts)

    # lf vs shf spatial dominance
    if land_mask_2d is not None:
        lf_dom_2d  = (lf_norm.reshape(refshape) > shf_norm.reshape(refshape)).mean(axis=0)
        S_mean_2d  = np.maximum(lf_norm, shf_norm).reshape(refshape).mean(axis=0)

    fig, axs = pplt.subplots([[1,2],[3,4]], figwidth=7.5, share=False)

    # (a) SR-HI analytic T–S grid
    density_ts, _ = np.histogram2d(T, S, bins=[Te, Se])
    m0 = axs[0].pcolormesh(np.linspace(T_lo,T_hi,NG), np.linspace(S_lo,S_hi,NG),
                            P_grid, cmap='Blues', vmin=0, vmax=vmax)
    axs[0].contour(Tcen, Scen, np.log10(density_ts.T.clip(1)),
                   levels=5, colors='k', linewidths=0.5, alpha=0.6)
    axs[0].format(xlabel='T (thermo.)', ylabel='S (surface)', title='SR-HI predicted P',
                  xlim=(T_lo,T_hi), ylim=(S_lo,S_hi))
    fig.colorbar(m0, ax=axs[0], loc='b', label='P (mm / 3 h)')

    # (b) Observed binned P in T–S space
    m1 = axs[1].pcolormesh(Tcen, Scen, obs_ts, cmap='Blues', vmin=0, vmax=vmax)
    axs[1].format(xlabel='T', ylabel='S', title='ERA5 observed (binned)',
                  xlim=(T_lo,T_hi), ylim=(S_lo,S_hi))
    fig.colorbar(m1, ax=axs[1], loc='b', label='P (mm / 3 h)')

    # (c) Partial dependence: P vs S at fixed T
    for perc, tv in zip(T_pcts, T_fixed):
        P_sw = to_phys((a_h*tv + d_h*S_sw)**3)
        axs[2].plot(S_sw, P_sw, label=f'T={perc}th pct', lw=1.3)
    axs[2].axhline(0, color='gray', lw=0.6)
    axs[2].format(xlabel='S', ylabel='P (mm / 3 h)', title='P vs S | fixed T')
    axs[2].legend(loc='ul', ncols=1, fontsize=7)

    # (d) LF vs SHF dominance map
    if land_mask_2d is not None:
        m3 = axs[3].pcolormesh(lons, lats, lf_dom_2d, cmap='RdBu_r', vmin=0, vmax=1)
        axs[3].format(xlabel='Lon', ylabel='Lat', title='Frac. time LF > SHF (controls S)')
        fig.colorbar(m3, ax=axs[3], loc='b', label='Fraction')
    else:
        axs[3].format(title='Land mask unavailable')

    fig.format(abc=True)
    pplt.show()
    fig.save('../figs/fig_srhi.pdf'); fig.save('../figs/fig_srhi.png')
    print(f'Suppressive term: d_h = {d_h:.4f}  (negative → S suppresses precipitation)')
    print(f'LF > SHF fraction: {(lf_norm[valid]>shf_norm[valid]).mean():.1%}')

In [ ]:
# ── Fig C: SR-LO  RH kernel + moisture transition ─────────────────────────────
if 'sr_lo' not in SR_REGISTRY:
    print('sr_lo not in registry — skipping')
else:
    _c = SR_REGISTRY['sr_lo']['constants']
    a_lo, b_lo = _c['a'], _c['b']

    rh_v = rh_k[valid]; P_v = true_flat[valid]
    rh_lo, rh_hi = prange(rh_v)

    # analytic curve
    rh_sw  = np.linspace(rh_lo, rh_hi, 300)
    P_lo   = to_phys(a_lo*np.exp(rh_sw) + b_lo)

    # 1-D binned observed
    _, rh_cen, obs_means, _ = bin1d(rh_v, P_v)
    # 1-D SR-LO prediction at bin centers
    pd_means = to_phys(a_lo*np.exp(rh_cen) + b_lo)

    fig, axs = pplt.subplots(ncols=3, figwidth=7.5, share=False)

    # (a) Learned RH kernel
    if avg_kernel is not None:
        k_rh = avg_kernel[0]                               # (nsig,)
        # show each seed for uncertainty
        for kw_seed in _kw_list:
            axs[0].plot(sig_hpa, kw_seed[0], color='gray', lw=0.7, alpha=0.5)
        axs[0].plot(sig_hpa, k_rh, color=COLORS['sr_lo'], lw=1.5)
        peak_p = sig_hpa[np.argmax(k_rh)]
        axs[0].axvline(peak_p, color='k', lw=0.8, ls='--', label=f'Peak {peak_p:.0f} hPa')
        axs[0].legend(loc='ul', ncols=1, fontsize=7)
        axs[0].format(xlabel='Pressure (hPa)', ylabel='Kernel weight',
                      title='Learned RH kernel', xlim=(levmin, levmax))
    else:
        axs[0].text(0.5, 0.5, 'Kernel weights unavailable', transform=axs[0].transAxes,
                    ha='center', va='center')

    # (b) Analytic PD curve
    axs[1].plot(rh_sw, P_lo, color=COLORS['sr_lo'], lw=1.5, label='SR-LO')
    axs[1].axhline(0, color='gray', lw=0.6)
    axs[1].format(xlabel=r'Kernel-integrated $\widehat{\mathrm{RH}}$',
                  ylabel='P (mm / 3 h)', title='SR-LO moisture transition')

    # (c) Binned observed vs SR-LO
    axs[2].scatter(rh_cen, obs_means, color='gray', s=15, label='ERA5 binned', zorder=4)
    axs[2].plot(rh_sw, P_lo, color=COLORS['sr_lo'], lw=1.5, label='SR-LO', zorder=5)
    axs[2].format(xlabel=r'Kernel-integrated $\widehat{\mathrm{RH}}$',
                  ylabel='P (mm / 3 h)', title='Observed vs SR-LO')
    axs[2].legend(loc='ul', ncols=1, fontsize=7)

    fig.format(abc=True, grid=False)
    pplt.show()
    fig.save('../figs/fig_srlo.pdf'); fig.save('../figs/fig_srlo.png')

In [ ]:
# ── Fig D: SR-BL vs POD-BL  nonlinear buoyancy transition ────────────────────
if 'sr_bl' not in SR_REGISTRY or bl_norm is None:
    print('sr_bl not in registry or bl unavailable — skipping')
else:
    _c = SR_REGISTRY['sr_bl']['constants']
    a_bl, b_bl = _c['a'], _c['b']
    bl_std = float(STATS.get('bl_std', 1.0)); bl_mean = float(STATS.get('bl_mean', 0.0))

    bl_v = bl_norm[valid]; P_v = true_flat[valid]
    bl_nat_v = bl_v * bl_std + bl_mean                     # native BL units

    bl_lo, bl_hi = prange(bl_nat_v)
    bl_sw_nat = np.linspace(bl_lo, bl_hi, 400)
    bl_sw_nrm = (bl_sw_nat - bl_mean) / bl_std

    # SR-BL analytic prediction
    P_srbl = to_phys((bl_sw_nrm + a_bl)**3 + b_bl)

    # POD-BL analytic prediction
    P_pod = pod_alpha * np.maximum(0.0, bl_sw_nat - pod_xcrit) if pod_alpha is not None else None

    # 1-D binned observed
    _, bl_cen, obs_means, _ = bin1d(bl_nat_v, P_v)

    # derivative dP/dBL
    dbl  = bl_sw_nat[1] - bl_sw_nat[0]
    dP_srbl = np.gradient(P_srbl, dbl)
    dP_pod  = np.gradient(P_pod, dbl) if P_pod is not None else None

    fig, axs = pplt.subplots(ncols=2, figwidth=6.0, share=False)

    # (a) Binned obs + models
    axs[0].scatter(bl_cen, obs_means, color='gray', s=15, zorder=4, label='ERA5 binned')
    axs[0].plot(bl_sw_nat, P_srbl, color=COLORS['sr_bl'], lw=1.5, label=LABELS['sr_bl'])
    if P_pod is not None:
        axs[0].plot(bl_sw_nat, P_pod, color=COLORS['pod_bl'], lw=1.5, ls='--', label=LABELS['pod_bl'])
    axs[0].axvline(pod_xcrit if pod_xcrit is not None else 0, color='k', lw=0.7, ls=':')
    axs[0].format(xlabel=r'$B_L$ (m s$^{-2}$)', ylabel='P (mm / 3 h)',
                  title='P vs BL: observed and learned', xlim=(bl_lo, bl_hi), ylim=(0, None))
    axs[0].legend(loc='ul', ncols=1, fontsize=7)

    # (b) Sensitivity dP/dBL
    axs[1].plot(bl_sw_nat, dP_srbl, color=COLORS['sr_bl'], lw=1.5, label=LABELS['sr_bl'])
    if dP_pod is not None:
        axs[1].plot(bl_sw_nat, dP_pod, color=COLORS['pod_bl'], lw=1.5, ls='--', label=LABELS['pod_bl'])
    axs[1].axhline(0, color='gray', lw=0.6)
    axs[1].format(xlabel=r'$B_L$ (m s$^{-2}$)', ylabel=r'dP/d$B_L$',
                  title='Sensitivity', xlim=(bl_lo, bl_hi))
    axs[1].legend(loc='ul', ncols=1, fontsize=7)

    fig.format(abc=True, grid=False)
    pplt.show()
    fig.save('../figs/fig_srbl.pdf'); fig.save('../figs/fig_srbl.png')

In [ ]:
# ── Fig E: Kernel summary — RH, θe, θe* ──────────────────────────────────────
if avg_kernel is None:
    print('No kernel weights available — skipping kernel summary')
else:
    var_labels = [r'$\widehat{\mathrm{RH}}$', r'$\hat{\theta}_e$', r'$\hat{\theta}_e^*$']
    var_colors = [COLORS.get('sr_lo','C0'), COLORS.get('sr_med','C1'), COLORS.get('nn_gauss','C2')]

    fig, axs = pplt.subplots(ncols=3, figwidth=6.5, sharey=True)
    for i, (lbl, col) in enumerate(zip(var_labels, var_colors)):
        k_mean = avg_kernel[i]
        # individual seeds as thin gray lines
        for kw_s in _kw_list:
            axs[i].plot(kw_s[i], sig_hpa, color='gray', lw=0.6, alpha=0.5)
        axs[i].plot(k_mean, sig_hpa, color=col, lw=2.0)
        peak_idx = np.argmax(k_mean)
        axs[i].scatter(k_mean[peak_idx], sig_hpa[peak_idx], color=col, s=40, zorder=6)
        axs[i].axhline(sig_hpa[peak_idx], color=col, lw=0.7, ls='--', alpha=0.7)
        axs[i].format(xlabel='Kernel weight', title=lbl, ylim=(levmax, levmin))  # inverted pressure
        if i == 0:
            axs[i].format(ylabel='Pressure (hPa)')

    fig.format(abc=True, grid=False)
    pplt.show()
    fig.save('../figs/fig_kernels.pdf'); fig.save('../figs/fig_kernels.png')

    print('Kernel peak pressures:')
    for i, lbl in enumerate(var_labels):
        k = avg_kernel[i]; pi = np.argmax(k)
        print(f'  {lbl}: peak at {sig_hpa[pi]:.0f} hPa  weight={k[pi]:.4f}')

In [ ]:
# ── Fig F: Spatial regime maps ────────────────────────────────────────────────
if 'sr_med' not in SR_REGISTRY or 'sr_hi' not in SR_REGISTRY:
    print('SR models missing — skipping spatial maps')
else:
    # Load SR-HI predictions from saved file
    _phi = os.path.join(PREDSDIR, f'sr_hi_{SPLIT}_predictions.nc')
    if os.path.exists(_phi):
        with xr.open_dataset(_phi, engine='h5netcdf') as _pds:
            _pred = _pds.tp
            if 'seed' in _pred.dims: _pred = _pred.mean('seed')
            if 'complexity' in _pred.dims: _pred = _pred.isel(complexity=0)
            srhi_pred_2d = _pred.transpose('time','lat','lon').values.mean(axis=0)
    else:
        # fall back to computing from registry
        _c = SR_REGISTRY['sr_hi']['constants']
        a_h2, b_h2, c_h2, d_h2 = _c['a'], _c['b'], _c['c'], _c['d']
        _M = rh_k; _I = thetae_k + b_h2*thetaestar_k + c_h2
        _T = np.maximum(_M, _I)
        _S = np.maximum(lf_norm, shf_norm) if (lf_norm is not None and shf_norm is not None) else np.zeros_like(_T)
        srhi_pred_2d = to_phys((a_h2*_T + d_h2*_S)**3).reshape(refshape).mean(axis=0)

    # True precip mean
    true_2d = true_flat.reshape(refshape)
    true_mean = np.nanmean(true_2d, axis=0)
    bias_2d   = srhi_pred_2d - true_mean

    # SR-MED regime fractions (per grid point)
    _c = SR_REGISTRY['sr_med']['constants']
    a_m2, b_m2, c_m2 = _c['a'], _c['b'], _c['c']
    M_3d = rh_k.reshape(refshape)
    I_3d = (thetae_k - b_m2*thetaestar_k - c_m2).reshape(refshape)
    frac_moist = (M_3d >= I_3d).mean(axis=0)
    frac_insta = 1.0 - frac_moist

    latlim = (float(lats.min()), float(lats.max()))
    lonlim = (float(lons.min()), float(lons.max()))

    fig, axs = pplt.subplots(nrows=2, ncols=3, proj='cyl', figwidth=7.5,
                              share=False, tight=True)
    kw_map = dict(coast=True, lonlim=lonlim, latlim=latlim, grid=False)

    vp = float(np.nanpercentile(true_mean, 97))
    m0 = axs[0].pcolormesh(lons, lats, true_mean, cmap='Blues', vmin=0, vmax=vp)
    axs[0].format(title='ERA5 mean P', **kw_map)
    fig.colorbar(m0, ax=axs[0], loc='b', label='mm / 3 h')

    m1 = axs[1].pcolormesh(lons, lats, srhi_pred_2d, cmap='Blues', vmin=0, vmax=vp)
    axs[1].format(title='SR-HI mean P', **kw_map)
    fig.colorbar(m1, ax=axs[1], loc='b', label='mm / 3 h')

    blim = float(np.nanpercentile(np.abs(bias_2d), 97))
    m2 = axs[2].pcolormesh(lons, lats, bias_2d, cmap='RdBu_r', vmin=-blim, vmax=blim)
    axs[2].format(title='Bias (SR-HI − ERA5)', **kw_map)
    fig.colorbar(m2, ax=axs[2], loc='b', label='mm / 3 h')

    m3 = axs[3].pcolormesh(lons, lats, frac_moist, cmap='Blues', vmin=0, vmax=1)
    axs[3].format(title='Frac. moisture-ctrl (M ≥ I)', **kw_map)
    fig.colorbar(m3, ax=axs[3], loc='b', label='Fraction')

    m4 = axs[4].pcolormesh(lons, lats, frac_insta, cmap='Reds', vmin=0, vmax=1)
    axs[4].format(title='Frac. instability-ctrl (I > M)', **kw_map)
    fig.colorbar(m4, ax=axs[4], loc='b', label='Fraction')

    if lf_norm is not None and shf_norm is not None:
        lf_dom_map = (lf_norm.reshape(refshape) > shf_norm.reshape(refshape)).mean(axis=0)
        m5 = axs[5].pcolormesh(lons, lats, lf_dom_map, cmap='RdBu_r', vmin=0, vmax=1)
        axs[5].format(title='Frac. LF > SHF controls S', **kw_map)
        fig.colorbar(m5, ax=axs[5], loc='b', label='Fraction')

    fig.format(abc=True)
    pplt.show()
    fig.save('../figs/fig_maps.pdf'); fig.save('../figs/fig_maps.png')

In [ ]:
# ── Fig G: Precipitation intensity diagnostics ────────────────────────────────
model_preds = {}
for name in ['pod_bl','sr_bl','sr_lo','sr_med','sr_hi','nn_gauss','nn_full']:
    _fp = os.path.join(PREDSDIR, f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(_fp): continue
    with xr.open_dataset(_fp, engine='h5netcdf') as _ds:
        _p = _ds.tp
        if 'seed' in _p.dims: _p = _p.mean('seed')
        if 'complexity' in _p.dims: _p = _p.isel(complexity=0)
        model_preds[name] = _p.transpose('time','lat','lon').values.ravel()

if not model_preds:
    print('No prediction files found — skipping intensity figure')
else:
    P_obs = true_flat[valid]
    pctiles = np.linspace(0, 100, NBINS+1)
    obs_quantiles = np.percentile(P_obs[P_obs > 0], pctiles)
    obs_edges  = np.percentile(P_obs, np.linspace(1, 99, NBINS+1))
    obs_cen    = 0.5*(obs_edges[:-1]+obs_edges[1:])

    fig, axs = pplt.subplots(ncols=2, figwidth=6.5, share=False)

    # (a) Quantile–quantile: observed vs predicted
    axs[0].plot([0, obs_quantiles.max()], [0, obs_quantiles.max()],
                'k--', lw=0.8, label='1:1')
    for name, pred in model_preds.items():
        pred_v = pred[valid]
        pred_q = np.percentile(pred_v[pred_v > 0], pctiles)
        axs[0].plot(obs_quantiles, pred_q, color=COLORS.get(name,'gray'),
                    lw=1.2, label=LABELS.get(name, name))
    axs[0].format(xlabel='Observed quantile (mm / 3 h)',
                  ylabel='Predicted quantile (mm / 3 h)', title='Q–Q plot')
    axs[0].legend(loc='ul', ncols=1, fontsize=6)

    # (b) Bias by observed precipitation bin
    axs[1].axhline(0, color='k', lw=0.7)
    for name, pred in model_preds.items():
        pred_v = pred[valid]
        _, cen, bias_means, _ = bin1d(P_obs, pred_v - P_obs, nbins=NBINS,
                                       minsamp=MINSAMP, plo=1, phi=99)
        axs[1].plot(cen, bias_means, color=COLORS.get(name,'gray'),
                    lw=1.2, label=LABELS.get(name, name))
    axs[1].format(xlabel='Observed P (mm / 3 h)', ylabel='Bias (pred − obs)',
                  title='Bias by precipitation bin')
    axs[1].legend(loc='ll', ncols=1, fontsize=6)

    fig.format(abc=True, grid=False)
    pplt.show()
    fig.save('../figs/fig_intensity.pdf'); fig.save('../figs/fig_intensity.png')